# 🌲 Tree-Based Models Training Pipeline

## Welcome to the Tree-Based ML Training Notebook!

This notebook provides a **complete workflow** for training tree-based models for stock market prediction. Tree-based models are excellent for:

- 📊 **Interpretability**: See which features matter most
- 🚀 **Fast Training**: Train quickly even on CPU
- 💪 **Robust**: Handle missing values and outliers well
- 🎯 **No Scaling Required**: Work with raw feature values

### Supported Models
| Model | Description | Best For |
|-------|-------------|----------|
| **Random Forest** | Ensemble of decision trees | Balanced accuracy & interpretability |
| **XGBoost** | Gradient boosted trees | Maximum predictive performance |

### How to Use This Notebook
1. **Run cells in order** (top to bottom)
2. **Green checkmarks** ✅ indicate completed steps
3. **Interactive widgets** let you configure without coding
4. **Checkpoints** are shared with the Neural Networks notebook!

### Comparison with Neural Networks
| Aspect | Tree-Based | Neural Networks |
|--------|------------|-----------------|
| Training Speed | ⚡ Fast | 🐢 Slower |
| Interpretability | ✅ High | ❌ Low |
| Data Requirements | Less data needed | More data needed |
| Feature Engineering | Less important | More important |

---
*💡 Tip: Start here if you're new to ML - tree models are easier to understand!*

## 📦 Section 1: First-Time Setup

This section handles the initial environment setup. **Run this once** when you first open the notebook.

### What happens here:
1. ✅ Install required packages (scikit-learn, XGBoost, etc.)
2. ✅ Mount Google Drive for checkpoints
3. ✅ Configure API keys for data sources
4. ✅ Verify everything is working

*Note: This setup is compatible with the Neural Networks notebook - they share the same checkpoint folder!*

In [ ]:
#@title 🔧 Step 1.1: Install Dependencies { display-mode: "form" }
#@markdown Click the **Run** button (▶️) to install all required packages.
#@markdown This may take 2-3 minutes on first run.

import subprocess
import sys

def install_packages():
    """Install required packages with progress tracking."""
    packages = [
        ("scikit-learn", "Scikit-learn - ML algorithms & utilities"),
        ("xgboost", "XGBoost - Gradient boosted trees"),
        ("pandas", "Pandas - Data manipulation"),
        ("numpy", "NumPy - Numerical computing"),
        ("yfinance", "yFinance - Yahoo Finance data"),
        ("matplotlib", "Matplotlib - Plotting"),
        ("seaborn", "Seaborn - Statistical visualization"),
        ("ipywidgets", "ipywidgets - Interactive widgets"),
        ("onnx", "ONNX - Model export format"),
        ("onnxruntime", "ONNX Runtime - Model validation"),
        ("skl2onnx", "skl2onnx - Scikit-learn to ONNX converter"),
        ("onnxmltools", "onnxmltools - XGBoost to ONNX converter"),
        ("optuna", "Optuna - Hyperparameter tuning"),
        ("wandb", "Weights & Biases - Experiment tracking"),
        ("h5py", "H5py - Checkpoint storage"),
        ("requests", "Requests - API calls"),
        ("tqdm", "tqdm - Progress bars"),
        ("shap", "SHAP - Feature importance explanations"),
    ]
    
    print("📦 Installing packages...\n")
    print("=" * 60)
    
    for i, (package, description) in enumerate(packages, 1):
        print(f"[{i}/{len(packages)}] Installing {package}...")
        print(f"    📝 {description}")
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", package],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            print(f"    ✅ Success!\n")
        except subprocess.CalledProcessError:
            print(f"    ⚠️ Warning: Could not install {package}")
            print(f"    💡 Try: !pip install {package}\n")
    
    print("=" * 60)
    print("\n✅ Package installation complete!")
    print("💡 If any package failed, you can install it manually above.")

# Run installation
install_packages()

In [ ]:
#@title 💾 Step 1.2: Mount Google Drive { display-mode: "form" }
#@markdown Your checkpoints, data, and exported models will be saved to Google Drive.
#@markdown **Same folder as Neural Networks notebook** - experiments are shared!

import os
from datetime import datetime

# Check if running in Colab
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    from google.colab import drive
    
    print("🔗 Mounting Google Drive...")
    print("   You may need to authorize access in the popup window.\n")
    
    try:
        drive.mount('/content/drive')
        
        # Create project directory structure (shared with Neural Networks notebook)
        DRIVE_BASE = '/content/drive/MyDrive/trading_ml'
        DIRS = {
            'checkpoints': f'{DRIVE_BASE}/checkpoints',
            'data': f'{DRIVE_BASE}/data',
            'models': f'{DRIVE_BASE}/models',
            'exports': f'{DRIVE_BASE}/exports',
            'logs': f'{DRIVE_BASE}/logs'
        }
        
        for name, path in DIRS.items():
            os.makedirs(path, exist_ok=True)
            print(f"✅ {name}: {path}")
        
        print("\n" + "=" * 60)
        print("✅ Google Drive mounted successfully!")
        print(f"📁 Project folder: {DRIVE_BASE}")
        print("🔗 Shared with Neural Networks notebook!")
        
    except Exception as e:
        print(f"❌ Error mounting Drive: {e}")
        print("💡 Try: Runtime → Restart runtime, then run this cell again")
else:
    # Local environment - use local folders
    DRIVE_BASE = './trading_ml_data'
    DIRS = {
        'checkpoints': f'{DRIVE_BASE}/checkpoints',
        'data': f'{DRIVE_BASE}/data',
        'models': f'{DRIVE_BASE}/models',
        'exports': f'{DRIVE_BASE}/exports',
        'logs': f'{DRIVE_BASE}/logs'
    }
    
    for name, path in DIRS.items():
        os.makedirs(path, exist_ok=True)
    
    print("📁 Running in local environment")
    print(f"📂 Data folder: {DRIVE_BASE}")
    print("✅ Directories created!")

In [ ]:
#@title 🔑 Step 1.3: Configure API Keys { display-mode: "form" }
#@markdown API keys allow access to premium data sources and experiment tracking.

import os

# Initialize API keys dictionary
API_KEYS = {}

def get_api_key(name, env_var, required=False):
    """Safely get API key from multiple sources."""
    key = None
    
    # Try Colab secrets first
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get(env_var)
        except:
            pass
    
    # Try environment variable
    if not key:
        key = os.environ.get(env_var)
    
    # Status message
    if key:
        masked = key[:4] + '*' * (len(key) - 8) + key[-4:] if len(key) > 8 else '****'
        print(f"✅ {name}: {masked}")
        return key
    else:
        status = "❌ Missing (Required)" if required else "⚪ Not set (Optional)"
        print(f"{status}: {name}")
        if required:
            print(f"   💡 Add '{env_var}' to Colab Secrets or set environment variable")
        return None

print("🔑 Checking API Keys...\n")
print("=" * 60)

# Check each API key
API_KEYS['alpha_vantage'] = get_api_key(
    "Alpha Vantage", 
    "ALPHA_VANTAGE_KEY",
    required=False
)

API_KEYS['dashboard'] = get_api_key(
    "Dashboard API", 
    "DASHBOARD_API_KEY",
    required=False
)

API_KEYS['wandb'] = get_api_key(
    "Weights & Biases",
    "WANDB_API_KEY", 
    required=False
)

print("=" * 60)
print("\n📋 Summary:")
print("   • Yahoo Finance: ✅ No API key needed (free)")
print("   • Alpha Vantage: " + ("✅ Configured" if API_KEYS['alpha_vantage'] else "⚪ Using Yahoo Finance instead"))
print("   • W&B Tracking:  " + ("✅ Configured" if API_KEYS['wandb'] else "⚪ Using local logging"))
print("   • Dashboard API: " + ("✅ Configured" if API_KEYS['dashboard'] else "⚪ Manual upload required"))

print("\n💡 You can proceed without optional API keys - we'll use free alternatives!")

In [ ]:
#@title ✅ Step 1.4: Verify Environment { display-mode: "form" }
#@markdown This cell runs health checks to ensure everything is working.

import numpy as np

def run_health_checks():
    """Run comprehensive environment health checks."""
    results = {}
    
    print("🔍 Running Health Checks...\n")
    print("=" * 60)
    
    # Check 1: Scikit-learn
    try:
        print("1️⃣ Scikit-learn Installation")
        import sklearn
        print(f"   Version: {sklearn.__version__}")
        results['sklearn'] = True
        print("   ✅ Working!\n")
    except Exception as e:
        results['sklearn'] = False
        print(f"   ❌ Error: {e}\n")
    
    # Check 2: XGBoost
    try:
        print("2️⃣ XGBoost Installation")
        import xgboost as xgb
        print(f"   Version: {xgb.__version__}")
        results['xgboost'] = True
        print("   ✅ Working!\n")
    except Exception as e:
        results['xgboost'] = False
        print(f"   ❌ Error: {e}\n")
    
    # Check 3: Data libraries
    print("3️⃣ Data Libraries")
    try:
        import pandas as pd
        import yfinance as yf
        print(f"   Pandas: {pd.__version__}")
        print(f"   yfinance: {yf.__version__}")
        results['data_libs'] = True
        print("   ✅ Ready!\n")
    except ImportError as e:
        results['data_libs'] = False
        print(f"   ❌ Missing: {e}\n")
    
    # Check 4: ONNX export tools
    print("4️⃣ ONNX Export Tools")
    try:
        import onnx
        import skl2onnx
        print(f"   ONNX: {onnx.__version__}")
        print(f"   skl2onnx: {skl2onnx.__version__}")
        results['onnx'] = True
        print("   ✅ Ready!\n")
    except ImportError as e:
        results['onnx'] = False
        print(f"   ⚠️ Missing: {e}")
        print(f"   💡 ONNX export may not work\n")
    
    # Check 5: Storage access
    print("5️⃣ Storage Access")
    try:
        test_file = os.path.join(DIRS['checkpoints'], '.test_write')
        with open(test_file, 'w') as f:
            f.write('test')
        os.remove(test_file)
        results['storage'] = True
        print(f"   📁 Checkpoint folder: {DIRS['checkpoints']}")
        print("   ✅ Read/write access confirmed!\n")
    except Exception as e:
        results['storage'] = False
        print(f"   ❌ Storage error: {e}\n")
    
    # Check 6: Network
    print("6️⃣ Network Connectivity")
    try:
        import urllib.request
        urllib.request.urlopen('https://finance.yahoo.com', timeout=5)
        results['network'] = True
        print("   ✅ Internet connection working!\n")
    except:
        results['network'] = False
        print("   ⚠️ Network issues detected\n")
    
    # Summary
    print("=" * 60)
    passed = sum(results.values())
    total = len(results)
    
    if passed == total:
        print(f"\n🎉 All {total} checks passed! You're ready to train.")
    elif passed >= total - 1:
        print(f"\n✅ {passed}/{total} checks passed. Ready to proceed!")
    else:
        print(f"\n⚠️ {passed}/{total} checks passed. Review warnings above.")
    
    return results

# Run the checks
health_results = run_health_checks()

## 🔄 Section 2: Session Resume

This section checks for previous work and lets you continue where you left off.

**Cross-Notebook Compatibility:**
- Checkpoints from Neural Networks notebook are visible here
- Data preprocessed in either notebook can be reused
- Experiments track which notebook was used

In [ ]:
#@title 🔍 Check for Previous Sessions { display-mode: "form" }
#@markdown Detects checkpoints from both this notebook AND the Neural Networks notebook.

import glob
import json
import h5py

class SessionManager:
    """Manages session state and checkpoints (shared across notebooks)."""
    
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = checkpoint_dir
        self.current_session = None
        self.notebook_type = 'tree_based'  # Identifies this notebook
        self.state = {
            'step': 'setup',
            'notebook': 'tree_based',
            'config': {},
            'data_loaded': False,
            'preprocessed': False,
            'models_trained': [],
            'timestamp': None
        }
    
    def find_checkpoints(self):
        """Find all available checkpoints (from both notebooks)."""
        pattern = os.path.join(self.checkpoint_dir, 'session_*.h5')
        checkpoints = glob.glob(pattern)
        
        if not checkpoints:
            return []
        
        # Sort by modification time (newest first)
        checkpoints.sort(key=os.path.getmtime, reverse=True)
        
        # Get metadata for each
        checkpoint_info = []
        for cp in checkpoints:
            try:
                with h5py.File(cp, 'r') as f:
                    info = {
                        'path': cp,
                        'timestamp': f.attrs.get('timestamp', 'Unknown'),
                        'step': f.attrs.get('step', 'Unknown'),
                        'notebook': f.attrs.get('notebook', 'unknown'),
                        'models': list(f.attrs.get('models_trained', [])),
                        'has_data': 'X_train' in f
                    }
                    checkpoint_info.append(info)
            except:
                continue
        
        return checkpoint_info
    
    def load_checkpoint(self, path):
        """Load state from checkpoint file."""
        with h5py.File(path, 'r') as f:
            self.state = {
                'step': f.attrs.get('step', 'setup'),
                'notebook': f.attrs.get('notebook', 'unknown'),
                'config': json.loads(f.attrs.get('config', '{}')),
                'data_loaded': f.attrs.get('data_loaded', False),
                'preprocessed': f.attrs.get('preprocessed', False),
                'models_trained': list(f.attrs.get('models_trained', [])),
                'timestamp': f.attrs.get('timestamp', None)
            }
            
            # Load data arrays if present
            self.data = {}
            for key in ['X_train', 'X_test', 'y_train', 'y_test']:
                if key in f:
                    self.data[key] = f[key][:]
        
        print(f"✅ Loaded checkpoint from {self.state['timestamp']}")
        return self.state
    
    def save_checkpoint(self, step, **kwargs):
        """Save current state to checkpoint."""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'session_{timestamp}.h5'
        path = os.path.join(self.checkpoint_dir, filename)
        
        self.state['step'] = step
        self.state['timestamp'] = timestamp
        self.state['notebook'] = self.notebook_type
        self.state.update(kwargs)
        
        with h5py.File(path, 'w') as f:
            # Save metadata
            f.attrs['step'] = step
            f.attrs['timestamp'] = timestamp
            f.attrs['notebook'] = self.notebook_type
            f.attrs['config'] = json.dumps(self.state.get('config', {}))
            f.attrs['data_loaded'] = self.state.get('data_loaded', False)
            f.attrs['preprocessed'] = self.state.get('preprocessed', False)
            f.attrs['models_trained'] = self.state.get('models_trained', [])
            
            # Save data arrays if available
            if hasattr(self, 'data'):
                for key, arr in self.data.items():
                    f.create_dataset(key, data=arr, compression='gzip')
        
        print(f"💾 Checkpoint saved: {filename}")
        return path

# Initialize session manager
session = SessionManager(DIRS['checkpoints'])

# Check for existing checkpoints
print("🔍 Checking for previous sessions...\n")
checkpoints = session.find_checkpoints()

if checkpoints:
    print("=" * 60)
    print("📋 Found previous sessions:\n")
    
    for i, cp in enumerate(checkpoints[:5], 1):
        notebook_icon = "🌲" if cp['notebook'] == 'tree_based' else "🧠"
        print(f"  {i}. {notebook_icon} {cp['timestamp']}")
        print(f"     Notebook: {cp['notebook']}")
        print(f"     Step: {cp['step']}")
        print(f"     Models: {len(cp['models'])}")
        print(f"     Has data: {'✅' if cp['has_data'] else '❌'}")
        print()
    
    print("=" * 60)
    print("\n🤔 Would you like to resume?")
    print("   • Run the next cell to RESUME from the latest checkpoint")
    print("   • Or skip to Section 3 to START FRESH")
    
    LATEST_CHECKPOINT = checkpoints[0]['path']
else:
    print("✨ No previous sessions found - starting fresh!")
    print("   Continue to Section 3 to configure your experiment.")
    LATEST_CHECKPOINT = None

In [ ]:
#@title 🔄 Resume from Checkpoint (Optional) { display-mode: "form" }
#@markdown **Only run this cell if you want to resume a previous session.**

if LATEST_CHECKPOINT:
    print("🔄 Resuming from checkpoint...\n")
    
    state = session.load_checkpoint(LATEST_CHECKPOINT)
    
    print("\n📊 Restored Session State:")
    print(f"   • Original notebook: {state['notebook']}")
    print(f"   • Last step: {state['step']}")
    print(f"   • Data loaded: {'✅' if state['data_loaded'] else '❌'}")
    print(f"   • Preprocessed: {'✅' if state['preprocessed'] else '❌'}")
    print(f"   • Models trained: {len(state['models_trained'])}")
    
    if state['config']:
        print(f"\n⚙️ Configuration:")
        for key, value in state['config'].items():
            print(f"   • {key}: {value}")
    
    # Load data if available
    if hasattr(session, 'data') and session.data:
        X_train = session.data.get('X_train')
        X_test = session.data.get('X_test')
        y_train = session.data.get('y_train')
        y_test = session.data.get('y_test')
        print(f"\n📊 Data loaded: {len(X_train) + len(X_test)} samples")
    
    print("\n✅ Session restored! Continue from the appropriate section.")
else:
    print("⚠️ No checkpoint to resume from.")
    print("   Continue to Section 3 to configure a new experiment.")

## ⚙️ Section 3: Experiment Configuration

Use the interactive widgets below to configure your experiment.

### Tree-Based Model Hyperparameters Explained:

| Parameter | Random Forest | XGBoost | Effect |
|-----------|---------------|---------|--------|
| **n_estimators** | Number of trees | Number of boosting rounds | More = better but slower |
| **max_depth** | Max tree depth | Max tree depth | Higher = more complex model |
| **min_samples_split** | Min samples to split | - | Higher = more regularization |
| **learning_rate** | - | Step size shrinkage | Lower = more conservative |
| **subsample** | - | Fraction of samples per tree | Lower = more regularization |

*💡 Tree-based models are less sensitive to hyperparameters than neural networks!*

In [ ]:
#@title 🎛️ Interactive Configuration Panel { display-mode: "form" }
#@markdown Adjust the settings below for your tree-based model experiment.

import ipywidgets as widgets
from IPython.display import display, HTML

# Store configuration
CONFIG = {}

# --- Basic Settings ---
experiment_name = widgets.Text(
    value=f'tree_exp_{datetime.now().strftime("%Y%m%d_%H%M")}',
    description='Experiment:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

model_select = widgets.SelectMultiple(
    options=['RandomForest', 'XGBoost'],
    value=['RandomForest'],
    description='Models:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px', height='60px')
)

verbose_mode = widgets.ToggleButtons(
    options=['Verbose', 'Quiet'],
    value='Verbose',
    description='Output:',
    style={'description_width': '120px'}
)

# --- Data Settings ---
data_source = widgets.Dropdown(
    options=[
        ('Yahoo Finance (Free)', 'yahoo'),
        ('Alpha Vantage (API Key)', 'alpha_vantage'),
        ('NSE/BSE India', 'nse_bse')
    ],
    value='yahoo',
    description='Data Source:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

ticker_input = widgets.Text(
    value='AAPL',
    description='Ticker(s):',
    placeholder='AAPL, MSFT, GOOGL',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

start_date = widgets.DatePicker(
    description='Start Date:',
    value=datetime(2020, 1, 1).date(),
    style={'description_width': '120px'}
)

end_date = widgets.DatePicker(
    description='End Date:',
    value=datetime(2024, 12, 31).date(),
    style={'description_width': '120px'}
)

train_split = widgets.FloatSlider(
    value=0.8,
    min=0.5,
    max=0.95,
    step=0.05,
    description='Train Split:',
    style={'description_width': '120px'},
    readout_format='.0%'
)

# --- Random Forest Parameters ---
rf_n_estimators = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description='n_estimators:',
    style={'description_width': '120px'}
)

rf_max_depth = widgets.IntSlider(
    value=10,
    min=3,
    max=30,
    step=1,
    description='max_depth:',
    style={'description_width': '120px'}
)

rf_min_samples_split = widgets.IntSlider(
    value=5,
    min=2,
    max=20,
    step=1,
    description='min_samples:',
    style={'description_width': '120px'}
)

# --- XGBoost Parameters ---
xgb_n_estimators = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description='n_estimators:',
    style={'description_width': '120px'}
)

xgb_max_depth = widgets.IntSlider(
    value=6,
    min=3,
    max=15,
    step=1,
    description='max_depth:',
    style={'description_width': '120px'}
)

xgb_learning_rate = widgets.SelectionSlider(
    options=[0.01, 0.05, 0.1, 0.2, 0.3],
    value=0.1,
    description='learning_rate:',
    style={'description_width': '120px'}
)

xgb_subsample = widgets.FloatSlider(
    value=0.8,
    min=0.5,
    max=1.0,
    step=0.1,
    description='subsample:',
    style={'description_width': '120px'}
)

# --- Cross-Validation ---
cv_folds = widgets.IntSlider(
    value=5,
    min=2,
    max=10,
    step=1,
    description='CV Folds:',
    style={'description_width': '120px'}
)

# --- Output Area ---
output = widgets.Output()

# ============================================================
# LAYOUT
# ============================================================

def header(text, emoji='📌'):
    return widgets.HTML(f'<h4 style="margin-top:15px;">{emoji} {text}</h4>')

# Build the configuration panel
config_panel = widgets.VBox([
    widgets.HTML('<h3>🎛️ Tree-Based Models Configuration</h3>'),
    widgets.HTML('<hr>'),
    
    header('Basic Settings', '📋'),
    experiment_name,
    model_select,
    verbose_mode,
    
    header('Data Settings', '📊'),
    data_source,
    ticker_input,
    widgets.HBox([start_date, end_date]),
    train_split,
    
    header('Random Forest Parameters', '🌲'),
    widgets.HTML('<small style="color:gray;">📝 More trees = better but slower. Max depth controls complexity.</small>'),
    rf_n_estimators,
    rf_max_depth,
    rf_min_samples_split,
    
    header('XGBoost Parameters', '🚀'),
    widgets.HTML('<small style="color:gray;">📝 Lower learning rate + more estimators = better generalization.</small>'),
    xgb_n_estimators,
    xgb_max_depth,
    xgb_learning_rate,
    xgb_subsample,
    
    header('Cross-Validation', '🔄'),
    widgets.HTML('<small style="color:gray;">📝 More folds = more reliable estimate but slower.</small>'),
    cv_folds,
    
    widgets.HTML('<hr>'),
    output
])

# Display the panel
display(config_panel)

# ============================================================
# SAVE CONFIGURATION
# ============================================================

def save_config():
    """Save current widget values to CONFIG."""
    global CONFIG
    CONFIG = {
        'experiment_name': experiment_name.value,
        'models': list(model_select.value),
        'verbose': verbose_mode.value == 'Verbose',
        'data_source': data_source.value,
        'tickers': [t.strip() for t in ticker_input.value.split(',')],
        'start_date': start_date.value.strftime('%Y-%m-%d'),
        'end_date': end_date.value.strftime('%Y-%m-%d'),
        'train_split': train_split.value,
        'cv_folds': cv_folds.value,
        # Random Forest params
        'rf_n_estimators': rf_n_estimators.value,
        'rf_max_depth': rf_max_depth.value,
        'rf_min_samples_split': rf_min_samples_split.value,
        # XGBoost params
        'xgb_n_estimators': xgb_n_estimators.value,
        'xgb_max_depth': xgb_max_depth.value,
        'xgb_learning_rate': xgb_learning_rate.value,
        'xgb_subsample': xgb_subsample.value,
    }
    return CONFIG

# Auto-save on widget change
all_widgets = [experiment_name, model_select, verbose_mode, data_source, ticker_input,
               start_date, end_date, train_split, cv_folds,
               rf_n_estimators, rf_max_depth, rf_min_samples_split,
               xgb_n_estimators, xgb_max_depth, xgb_learning_rate, xgb_subsample]

for w in all_widgets:
    w.observe(lambda _: save_config(), names='value')

# Initial save
CONFIG = save_config()

with output:
    print("✅ Configuration panel loaded!")
    print("   Adjust settings above, then run the next cell to confirm.")

In [ ]:
#@title ✅ Confirm Configuration { display-mode: "form" }
#@markdown Run this cell to validate and lock your configuration.

from IPython.display import display, HTML, Markdown
import json

def validate_config(config):
    """Validate configuration settings."""
    errors = []
    warnings = []
    
    # Check experiment name
    if not config.get('experiment_name') or len(config['experiment_name']) < 3:
        errors.append("❌ Experiment name must be at least 3 characters")
    
    # Check models selected
    if not config.get('models'):
        errors.append("❌ At least one model must be selected")
    
    # Check tickers
    if not config.get('tickers') or config['tickers'] == ['']:
        errors.append("❌ At least one ticker symbol is required")
    
    # Check dates
    from datetime import datetime
    try:
        start = datetime.strptime(config['start_date'], '%Y-%m-%d')
        end = datetime.strptime(config['end_date'], '%Y-%m-%d')
        if start >= end:
            errors.append("❌ End date must be after start date")
        if (end - start).days < 365:
            warnings.append("⚠️ Less than 1 year of data - may lead to poor model performance")
    except:
        errors.append("❌ Invalid date format")
    
    # Check hyperparameters
    if config.get('rf_n_estimators', 0) < 10:
        warnings.append("⚠️ Very few Random Forest trees - consider increasing")
    
    if config.get('xgb_learning_rate', 0) > 0.3:
        warnings.append("⚠️ High XGBoost learning rate may cause overfitting")
    
    return errors, warnings

# Validate
errors, warnings = validate_config(CONFIG)

# Display results
print("=" * 60)
print("📋 CONFIGURATION SUMMARY")
print("=" * 60)

# Basic info
print(f"\n🔬 Experiment: {CONFIG['experiment_name']}")
print(f"🌳 Models: {', '.join(CONFIG['models'])}")
print(f"📊 Data: {CONFIG['data_source']} - {', '.join(CONFIG['tickers'])}")
print(f"📅 Period: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"🔀 Train/Test Split: {CONFIG['train_split']:.0%}/{1-CONFIG['train_split']:.0%}")
print(f"🔄 Cross-Validation: {CONFIG['cv_folds']} folds")

if 'RandomForest' in CONFIG['models']:
    print(f"\n🌲 Random Forest:")
    print(f"   • n_estimators: {CONFIG['rf_n_estimators']}")
    print(f"   • max_depth: {CONFIG['rf_max_depth']}")
    print(f"   • min_samples_split: {CONFIG['rf_min_samples_split']}")

if 'XGBoost' in CONFIG['models']:
    print(f"\n🚀 XGBoost:")
    print(f"   • n_estimators: {CONFIG['xgb_n_estimators']}")
    print(f"   • max_depth: {CONFIG['xgb_max_depth']}")
    print(f"   • learning_rate: {CONFIG['xgb_learning_rate']}")
    print(f"   • subsample: {CONFIG['xgb_subsample']}")

print("\n" + "=" * 60)

# Show errors and warnings
if errors:
    print("\n🚨 ERRORS (must fix before proceeding):")
    for e in errors:
        print(f"   {e}")

if warnings:
    print("\n⚠️ WARNINGS (consider reviewing):")
    for w in warnings:
        print(f"   {w}")

if not errors:
    CONFIG_LOCKED = True
    print("\n✅ Configuration validated! Ready to proceed.")
    print("   ➡️ Run the next section to fetch and preprocess data.")
else:
    CONFIG_LOCKED = False
    print("\n❌ Please fix errors above and run this cell again.")

---

## 📊 Section 4: Data Acquisition & Preprocessing

This section fetches market data and prepares features for tree-based models.

### Features We'll Create:

| Feature Category | Examples | Why It Helps |
|-----------------|----------|--------------|
| **Price-Based** | Returns, Log Returns, Price Change | Basic price movement signals |
| **Moving Averages** | SMA, EMA (5, 10, 20, 50, 200 day) | Trend identification |
| **Momentum** | RSI, MACD, ROC | Overbought/oversold signals |
| **Volatility** | ATR, Bollinger Bands, Std Dev | Risk measurement |
| **Volume** | OBV, Volume SMA, Volume Ratio | Confirmation of price moves |
| **Lag Features** | 1-5 day lagged returns | Historical patterns |

> 💡 **For Beginners**: Tree-based models excel at finding non-linear relationships between features. Unlike neural networks, they don't need feature scaling!

In [ ]:
#@title 📥 Fetch Market Data { display-mode: "form" }
#@markdown Fetches historical data with rate limiting for API protection.

import yfinance as yf
import pandas as pd
import numpy as np
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class DataFetcher:
    """Fetches market data with rate limiting and retry logic."""
    
    def __init__(self, source='yahoo', rate_limit=0.5):
        self.source = source
        self.rate_limit = rate_limit
        self.last_request = 0
        
    def _rate_limit_wait(self):
        """Wait to respect rate limits."""
        elapsed = time.time() - self.last_request
        if elapsed < self.rate_limit:
            time.sleep(self.rate_limit - elapsed)
        self.last_request = time.time()
    
    def fetch_yahoo(self, ticker, start, end, retries=3):
        """Fetch data from Yahoo Finance with exponential backoff."""
        for attempt in range(retries):
            try:
                self._rate_limit_wait()
                df = yf.download(ticker, start=start, end=end, progress=False)
                if len(df) > 0:
                    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
                    return df
            except Exception as e:
                wait_time = 2 ** attempt
                print(f"   ⚠️ Attempt {attempt+1} failed for {ticker}, retrying in {wait_time}s...")
                time.sleep(wait_time)
        return None
    
    def fetch(self, tickers, start, end):
        """Fetch data for multiple tickers."""
        all_data = {}
        total = len(tickers)
        
        for i, ticker in enumerate(tickers, 1):
            print(f"   📊 Fetching {ticker}... ({i}/{total})")
            
            if self.source == 'yahoo':
                df = self.fetch_yahoo(ticker, start, end)
            else:
                # For other sources, fall back to Yahoo
                df = self.fetch_yahoo(ticker, start, end)
            
            if df is not None and len(df) > 0:
                all_data[ticker] = df
                print(f"      ✅ Got {len(df)} rows")
            else:
                print(f"      ❌ Failed to fetch {ticker}")
        
        return all_data

# Fetch the data
print("=" * 60)
print("📥 FETCHING MARKET DATA")
print("=" * 60)

fetcher = DataFetcher(source=CONFIG['data_source'])
RAW_DATA = fetcher.fetch(
    CONFIG['tickers'],
    CONFIG['start_date'],
    CONFIG['end_date']
)

if RAW_DATA:
    print("\n✅ Data fetching complete!")
    print(f"   • Tickers loaded: {list(RAW_DATA.keys())}")
    for ticker, df in RAW_DATA.items():
        print(f"   • {ticker}: {len(df)} rows, {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")
else:
    print("\n❌ No data fetched! Check ticker symbols and try again.")

In [ ]:
#@title 🔧 Feature Engineering { display-mode: "form" }
#@markdown Creates technical indicators and features for tree-based models.

import pandas as pd
import numpy as np
from typing import Dict, Tuple

class FeatureEngineer:
    """Creates features for tree-based models."""
    
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.feature_names = []
    
    def _log(self, msg):
        if self.verbose:
            print(msg)
    
    def add_returns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add return-based features."""
        df = df.copy()
        df['returns'] = df['Close'].pct_change()
        df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
        
        # Lagged returns
        for lag in [1, 2, 3, 5]:
            df[f'returns_lag_{lag}'] = df['returns'].shift(lag)
        
        return df
    
    def add_moving_averages(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add moving average features."""
        df = df.copy()
        
        # Simple Moving Averages
        for window in [5, 10, 20, 50]:
            df[f'sma_{window}'] = df['Close'].rolling(window=window).mean()
            df[f'sma_{window}_ratio'] = df['Close'] / df[f'sma_{window}']
        
        # Exponential Moving Averages
        for span in [12, 26]:
            df[f'ema_{span}'] = df['Close'].ewm(span=span, adjust=False).mean()
        
        # Moving average crossovers
        df['sma_5_20_cross'] = (df['sma_5'] > df['sma_20']).astype(int)
        df['ema_cross'] = (df['ema_12'] > df['ema_26']).astype(int)
        
        return df
    
    def add_momentum(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add momentum indicators."""
        df = df.copy()
        
        # RSI
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))
        
        # MACD
        df['macd'] = df['ema_12'] - df['ema_26']
        df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
        df['macd_hist'] = df['macd'] - df['macd_signal']
        
        # Rate of Change
        for period in [5, 10, 20]:
            df[f'roc_{period}'] = ((df['Close'] - df['Close'].shift(period)) / 
                                   df['Close'].shift(period)) * 100
        
        return df
    
    def add_volatility(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add volatility features."""
        df = df.copy()
        
        # Rolling standard deviation
        for window in [5, 10, 20]:
            df[f'volatility_{window}'] = df['returns'].rolling(window=window).std()
        
        # Average True Range (ATR)
        high_low = df['High'] - df['Low']
        high_close = np.abs(df['High'] - df['Close'].shift())
        low_close = np.abs(df['Low'] - df['Close'].shift())
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        df['atr'] = tr.rolling(window=14).mean()
        
        # Bollinger Bands
        df['bb_middle'] = df['Close'].rolling(window=20).mean()
        bb_std = df['Close'].rolling(window=20).std()
        df['bb_upper'] = df['bb_middle'] + 2 * bb_std
        df['bb_lower'] = df['bb_middle'] - 2 * bb_std
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
        df['bb_position'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
        
        return df
    
    def add_volume_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add volume-based features."""
        df = df.copy()
        
        # Volume moving averages
        df['volume_sma_5'] = df['Volume'].rolling(window=5).mean()
        df['volume_sma_20'] = df['Volume'].rolling(window=20).mean()
        df['volume_ratio'] = df['Volume'] / df['volume_sma_20']
        
        # On-Balance Volume
        obv = (np.sign(df['Close'].diff()) * df['Volume']).cumsum()
        df['obv'] = obv
        df['obv_sma'] = obv.rolling(window=20).mean()
        
        return df
    
    def add_target(self, df: pd.DataFrame, horizon=1) -> pd.DataFrame:
        """Add prediction target (next day return direction)."""
        df = df.copy()
        
        # Binary target: 1 if next day return is positive, 0 otherwise
        df['target'] = (df['Close'].shift(-horizon) > df['Close']).astype(int)
        
        # Also add continuous target for regression
        df['target_returns'] = df['Close'].pct_change().shift(-horizon)
        
        return df
    
    def engineer_features(self, df: pd.DataFrame, target_horizon=1) -> pd.DataFrame:
        """Run the full feature engineering pipeline."""
        self._log("🔧 Engineering features...")
        
        self._log("   • Adding return features...")
        df = self.add_returns(df)
        
        self._log("   • Adding moving averages...")
        df = self.add_moving_averages(df)
        
        self._log("   • Adding momentum indicators...")
        df = self.add_momentum(df)
        
        self._log("   • Adding volatility features...")
        df = self.add_volatility(df)
        
        self._log("   • Adding volume features...")
        df = self.add_volume_features(df)
        
        self._log("   • Adding target variable...")
        df = self.add_target(df, horizon=target_horizon)
        
        # Drop rows with NaN
        initial_len = len(df)
        df = df.dropna()
        dropped = initial_len - len(df)
        self._log(f"   • Dropped {dropped} rows with NaN values")
        
        # Store feature names (exclude OHLCV and targets)
        exclude_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 
                        'target', 'target_returns']
        self.feature_names = [c for c in df.columns if c not in exclude_cols]
        
        return df

# Process each ticker
print("=" * 60)
print("🔧 FEATURE ENGINEERING")
print("=" * 60)

engineer = FeatureEngineer(verbose=CONFIG['verbose'])
PROCESSED_DATA = {}

for ticker, raw_df in RAW_DATA.items():
    print(f"\n📊 Processing {ticker}...")
    processed = engineer.engineer_features(raw_df.copy())
    PROCESSED_DATA[ticker] = processed
    print(f"   ✅ {len(processed)} samples, {len(engineer.feature_names)} features")

# Show feature list
print("\n" + "=" * 60)
print("📋 FEATURES CREATED:")
print("=" * 60)
for i, feat in enumerate(engineer.feature_names, 1):
    print(f"   {i:2d}. {feat}")

print(f"\n✅ Feature engineering complete!")
print(f"   Total features: {len(engineer.feature_names)}")

In [ ]:
#@title ✂️ Train/Test Split { display-mode: "form" }
#@markdown Splits data into training and test sets (time-series aware).

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

def prepare_train_test(data: dict, feature_names: list, train_ratio: float):
    """
    Prepare train/test sets from processed data.
    Uses time-based split to prevent data leakage.
    """
    datasets = {}
    
    for ticker, df in data.items():
        # Get features and target
        X = df[feature_names].values
        y = df['target'].values
        
        # Time-based split (not random!)
        split_idx = int(len(df) * train_ratio)
        
        X_train = X[:split_idx]
        X_test = X[split_idx:]
        y_train = y[:split_idx]
        y_test = y[split_idx:]
        
        # Also keep dates for later analysis
        dates_train = df.index[:split_idx]
        dates_test = df.index[split_idx:]
        
        datasets[ticker] = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test,
            'dates_train': dates_train,
            'dates_test': dates_test,
            'feature_names': feature_names
        }
    
    return datasets

# Prepare datasets
print("=" * 60)
print("✂️ PREPARING TRAIN/TEST SPLIT")
print("=" * 60)

DATASETS = prepare_train_test(
    PROCESSED_DATA, 
    engineer.feature_names, 
    CONFIG['train_split']
)

for ticker, ds in DATASETS.items():
    print(f"\n📊 {ticker}:")
    print(f"   • Training samples: {len(ds['X_train'])} ({ds['dates_train'][0].strftime('%Y-%m-%d')} to {ds['dates_train'][-1].strftime('%Y-%m-%d')})")
    print(f"   • Test samples: {len(ds['X_test'])} ({ds['dates_test'][0].strftime('%Y-%m-%d')} to {ds['dates_test'][-1].strftime('%Y-%m-%d')})")
    print(f"   • Train class balance: {ds['y_train'].mean():.1%} positive")
    print(f"   • Test class balance: {ds['y_test'].mean():.1%} positive")

print("\n✅ Data split complete!")

---

## 🏋️ Section 5: Model Training

Now we'll train the tree-based models with cross-validation.

### Why Cross-Validation?

Cross-validation helps us:
1. **Estimate Performance**: Get reliable accuracy estimates
2. **Detect Overfitting**: Large gap between train/CV score = overfitting
3. **Model Selection**: Compare models fairly

### Time-Series Cross-Validation

We use **TimeSeriesSplit** instead of random K-Fold because:
- Respects temporal order (no future data leakage)
- Each fold trains on past, validates on future
- More realistic estimate of real-world performance

```
Fold 1: [===train===] [val]
Fold 2: [====train====] [val]
Fold 3: [=====train=====] [val]
```

> 💡 **For Beginners**: Unlike neural networks, tree models train very fast! You can experiment more freely.

In [ ]:
#@title 🌲 Train Random Forest { display-mode: "form" }
#@markdown Trains Random Forest classifier with cross-validation.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import time

class RandomForestTrainer:
    """Trains and evaluates Random Forest models."""
    
    def __init__(self, n_estimators=100, max_depth=10, min_samples_split=5, 
                 cv_folds=5, verbose=True):
        self.model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1,  # Use all cores
            class_weight='balanced'  # Handle class imbalance
        )
        self.cv_folds = cv_folds
        self.verbose = verbose
        self.cv_scores = None
        self.train_time = None
        
    def _log(self, msg):
        if self.verbose:
            print(msg)
    
    def cross_validate(self, X, y):
        """Perform time-series cross-validation."""
        self._log("   🔄 Running cross-validation...")
        
        tscv = TimeSeriesSplit(n_splits=self.cv_folds)
        self.cv_scores = cross_val_score(self.model, X, y, cv=tscv, scoring='accuracy')
        
        self._log(f"   📊 CV Scores: {[f'{s:.3f}' for s in self.cv_scores]}")
        self._log(f"   📈 Mean CV Accuracy: {self.cv_scores.mean():.3f} (+/- {self.cv_scores.std()*2:.3f})")
        
        return self.cv_scores
    
    def train(self, X_train, y_train, X_test, y_test):
        """Train the model and evaluate."""
        self._log("   🏋️ Training Random Forest...")
        
        start_time = time.time()
        self.model.fit(X_train, y_train)
        self.train_time = time.time() - start_time
        
        self._log(f"   ⏱️ Training time: {self.train_time:.2f} seconds")
        
        # Evaluate
        train_pred = self.model.predict(X_train)
        test_pred = self.model.predict(X_test)
        
        metrics = {
            'train_accuracy': accuracy_score(y_train, train_pred),
            'test_accuracy': accuracy_score(y_test, test_pred),
            'test_precision': precision_score(y_test, test_pred, zero_division=0),
            'test_recall': recall_score(y_test, test_pred, zero_division=0),
            'test_f1': f1_score(y_test, test_pred, zero_division=0),
            'cv_mean': self.cv_scores.mean() if self.cv_scores is not None else None,
            'cv_std': self.cv_scores.std() if self.cv_scores is not None else None,
            'train_time': self.train_time
        }
        
        return metrics, test_pred

# Train Random Forest if selected
if 'RandomForest' in CONFIG['models']:
    print("=" * 60)
    print("🌲 TRAINING RANDOM FOREST")
    print("=" * 60)
    
    RF_MODELS = {}
    RF_METRICS = {}
    RF_PREDICTIONS = {}
    
    for ticker, ds in DATASETS.items():
        print(f"\n📊 {ticker}:")
        
        trainer = RandomForestTrainer(
            n_estimators=CONFIG['rf_n_estimators'],
            max_depth=CONFIG['rf_max_depth'],
            min_samples_split=CONFIG['rf_min_samples_split'],
            cv_folds=CONFIG['cv_folds'],
            verbose=CONFIG['verbose']
        )
        
        # Cross-validate first
        trainer.cross_validate(ds['X_train'], ds['y_train'])
        
        # Train and evaluate
        metrics, predictions = trainer.train(
            ds['X_train'], ds['y_train'],
            ds['X_test'], ds['y_test']
        )
        
        RF_MODELS[ticker] = trainer.model
        RF_METRICS[ticker] = metrics
        RF_PREDICTIONS[ticker] = predictions
        
        print(f"\n   📈 Results:")
        print(f"      • Train Accuracy: {metrics['train_accuracy']:.3f}")
        print(f"      • Test Accuracy:  {metrics['test_accuracy']:.3f}")
        print(f"      • Test Precision: {metrics['test_precision']:.3f}")
        print(f"      • Test Recall:    {metrics['test_recall']:.3f}")
        print(f"      • Test F1 Score:  {metrics['test_f1']:.3f}")
    
    print("\n✅ Random Forest training complete!")
else:
    print("⏭️ Random Forest not selected, skipping...")

In [ ]:
#@title 🚀 Train XGBoost { display-mode: "form" }
#@markdown Trains XGBoost classifier with cross-validation.

import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import time

class XGBoostTrainer:
    """Trains and evaluates XGBoost models."""
    
    def __init__(self, n_estimators=100, max_depth=6, learning_rate=0.1, 
                 subsample=0.8, cv_folds=5, verbose=True):
        self.model = xgb.XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            use_label_encoder=False,
            eval_metric='logloss',
            verbosity=0
        )
        self.cv_folds = cv_folds
        self.verbose = verbose
        self.cv_scores = None
        self.train_time = None
        
    def _log(self, msg):
        if self.verbose:
            print(msg)
    
    def cross_validate(self, X, y):
        """Perform time-series cross-validation."""
        self._log("   🔄 Running cross-validation...")
        
        tscv = TimeSeriesSplit(n_splits=self.cv_folds)
        self.cv_scores = cross_val_score(self.model, X, y, cv=tscv, scoring='accuracy')
        
        self._log(f"   📊 CV Scores: {[f'{s:.3f}' for s in self.cv_scores]}")
        self._log(f"   📈 Mean CV Accuracy: {self.cv_scores.mean():.3f} (+/- {self.cv_scores.std()*2:.3f})")
        
        return self.cv_scores
    
    def train(self, X_train, y_train, X_test, y_test):
        """Train the model with early stopping."""
        self._log("   🏋️ Training XGBoost...")
        
        start_time = time.time()
        
        # Use eval set for early stopping info (but don't actually stop early here)
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )
        
        self.train_time = time.time() - start_time
        self._log(f"   ⏱️ Training time: {self.train_time:.2f} seconds")
        
        # Evaluate
        train_pred = self.model.predict(X_train)
        test_pred = self.model.predict(X_test)
        
        metrics = {
            'train_accuracy': accuracy_score(y_train, train_pred),
            'test_accuracy': accuracy_score(y_test, test_pred),
            'test_precision': precision_score(y_test, test_pred, zero_division=0),
            'test_recall': recall_score(y_test, test_pred, zero_division=0),
            'test_f1': f1_score(y_test, test_pred, zero_division=0),
            'cv_mean': self.cv_scores.mean() if self.cv_scores is not None else None,
            'cv_std': self.cv_scores.std() if self.cv_scores is not None else None,
            'train_time': self.train_time
        }
        
        return metrics, test_pred

# Train XGBoost if selected
if 'XGBoost' in CONFIG['models']:
    print("=" * 60)
    print("🚀 TRAINING XGBOOST")
    print("=" * 60)
    
    XGB_MODELS = {}
    XGB_METRICS = {}
    XGB_PREDICTIONS = {}
    
    for ticker, ds in DATASETS.items():
        print(f"\n📊 {ticker}:")
        
        trainer = XGBoostTrainer(
            n_estimators=CONFIG['xgb_n_estimators'],
            max_depth=CONFIG['xgb_max_depth'],
            learning_rate=CONFIG['xgb_learning_rate'],
            subsample=CONFIG['xgb_subsample'],
            cv_folds=CONFIG['cv_folds'],
            verbose=CONFIG['verbose']
        )
        
        # Cross-validate first
        trainer.cross_validate(ds['X_train'], ds['y_train'])
        
        # Train and evaluate
        metrics, predictions = trainer.train(
            ds['X_train'], ds['y_train'],
            ds['X_test'], ds['y_test']
        )
        
        XGB_MODELS[ticker] = trainer.model
        XGB_METRICS[ticker] = metrics
        XGB_PREDICTIONS[ticker] = predictions
        
        print(f"\n   📈 Results:")
        print(f"      • Train Accuracy: {metrics['train_accuracy']:.3f}")
        print(f"      • Test Accuracy:  {metrics['test_accuracy']:.3f}")
        print(f"      • Test Precision: {metrics['test_precision']:.3f}")
        print(f"      • Test Recall:    {metrics['test_recall']:.3f}")
        print(f"      • Test F1 Score:  {metrics['test_f1']:.3f}")
    
    print("\n✅ XGBoost training complete!")
else:
    XGB_MODELS = {}
    XGB_METRICS = {}
    XGB_PREDICTIONS = {}
    print("⏭️ XGBoost not selected, skipping...")

In [ ]:
#@title 💾 Save Training Checkpoint { display-mode: "form" }
#@markdown Saves current progress to Google Drive (cross-notebook compatible).

import h5py
import json
import numpy as np
from datetime import datetime
import pickle

def save_checkpoint(
    experiment_name: str,
    config: dict,
    models: dict,
    metrics: dict,
    datasets: dict,
    feature_names: list,
    checkpoint_dir: str
):
    """
    Save training checkpoint in HDF5 format.
    Compatible with neural networks notebook checkpoints.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = f"{checkpoint_dir}/{experiment_name}_{timestamp}.h5"
    
    print(f"   💾 Saving to: {checkpoint_path}")
    
    with h5py.File(checkpoint_path, 'w') as f:
        # Metadata
        meta = f.create_group('metadata')
        meta.attrs['experiment_name'] = experiment_name
        meta.attrs['timestamp'] = timestamp
        meta.attrs['notebook_type'] = 'tree_based'  # Cross-notebook identification
        meta.attrs['config'] = json.dumps(config)
        
        # Feature names
        f.create_dataset('feature_names', data=np.array(feature_names, dtype='S'))
        
        # Models (serialize with pickle, store as bytes)
        models_grp = f.create_group('models')
        for model_type, model_dict in models.items():
            type_grp = models_grp.create_group(model_type)
            for ticker, model in model_dict.items():
                model_bytes = pickle.dumps(model)
                type_grp.create_dataset(ticker, data=np.void(model_bytes))
        
        # Metrics
        metrics_grp = f.create_group('metrics')
        for model_type, metrics_dict in metrics.items():
            type_grp = metrics_grp.create_group(model_type)
            for ticker, m in metrics_dict.items():
                ticker_grp = type_grp.create_group(ticker)
                for k, v in m.items():
                    if v is not None:
                        ticker_grp.attrs[k] = v
        
        # Datasets (feature arrays)
        data_grp = f.create_group('datasets')
        for ticker, ds in datasets.items():
            ticker_grp = data_grp.create_group(ticker)
            ticker_grp.create_dataset('X_train', data=ds['X_train'], compression='gzip')
            ticker_grp.create_dataset('X_test', data=ds['X_test'], compression='gzip')
            ticker_grp.create_dataset('y_train', data=ds['y_train'])
            ticker_grp.create_dataset('y_test', data=ds['y_test'])
    
    return checkpoint_path

# Save checkpoint
print("=" * 60)
print("💾 SAVING CHECKPOINT")
print("=" * 60)

# Collect all models and metrics
all_models = {}
all_metrics = {}

if 'RandomForest' in CONFIG['models'] and 'RF_MODELS' in dir():
    all_models['RandomForest'] = RF_MODELS
    all_metrics['RandomForest'] = RF_METRICS

if 'XGBoost' in CONFIG['models'] and 'XGB_MODELS' in dir():
    all_models['XGBoost'] = XGB_MODELS
    all_metrics['XGBoost'] = XGB_METRICS

if all_models:
    checkpoint_path = save_checkpoint(
        experiment_name=CONFIG['experiment_name'],
        config=CONFIG,
        models=all_models,
        metrics=all_metrics,
        datasets=DATASETS,
        feature_names=engineer.feature_names,
        checkpoint_dir=CHECKPOINT_DIR
    )
    
    print(f"\n✅ Checkpoint saved!")
    print(f"   📁 Path: {checkpoint_path}")
    print(f"   💡 You can resume from this checkpoint in either notebook.")
else:
    print("⚠️ No models trained yet. Train models first.")

---

## 🔍 Section 6: Feature Importance Analysis

One major advantage of tree-based models is **interpretability**. We can understand *why* the model makes predictions.

### Feature Importance Methods:

| Method | Description | Use When |
|--------|-------------|----------|
| **Built-in Importance** | How often a feature is used for splitting | Quick overview |
| **Permutation Importance** | Drop in accuracy when feature is shuffled | Reliable, model-agnostic |
| **SHAP Values** | Game-theoretic feature attribution | Detailed analysis |

> 💡 **For Beginners**: SHAP (SHapley Additive exPlanations) tells you how much each feature pushed the prediction up or down. Positive SHAP = pushed toward "Buy", Negative SHAP = pushed toward "Sell".

In [ ]:
#@title 📊 Built-in Feature Importance { display-mode: "form" }
#@markdown Visualizes which features the models use most.

import matplotlib.pyplot as plt
import numpy as np

def plot_feature_importance(model, feature_names, model_name, ticker, top_n=15):
    """Plot top N most important features."""
    importances = model.feature_importances_
    indices = np.argsort(importances)[-top_n:]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
    ax.barh(range(top_n), importances[indices], color=colors)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in indices])
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{model_name} - Top {top_n} Features ({ticker})')
    
    plt.tight_layout()
    return fig

# Plot for all trained models
print("=" * 60)
print("📊 FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

IMPORTANCE_FIGURES = {}

# Random Forest importance
if 'RandomForest' in CONFIG['models'] and RF_MODELS:
    print("\n🌲 Random Forest Feature Importance:")
    for ticker, model in RF_MODELS.items():
        fig = plot_feature_importance(
            model, 
            engineer.feature_names, 
            'Random Forest', 
            ticker
        )
        IMPORTANCE_FIGURES[f'RF_{ticker}'] = fig
        plt.show()

# XGBoost importance
if 'XGBoost' in CONFIG['models'] and XGB_MODELS:
    print("\n🚀 XGBoost Feature Importance:")
    for ticker, model in XGB_MODELS.items():
        fig = plot_feature_importance(
            model, 
            engineer.feature_names, 
            'XGBoost', 
            ticker
        )
        IMPORTANCE_FIGURES[f'XGB_{ticker}'] = fig
        plt.show()

print("\n✅ Feature importance visualizations complete!")

In [ ]:
#@title 🔬 SHAP Analysis { display-mode: "form" }
#@markdown Deep dive into feature contributions using SHAP values.

import shap
import matplotlib.pyplot as plt
import numpy as np

def run_shap_analysis(model, X_sample, feature_names, model_name, ticker, max_samples=500):
    """Run SHAP analysis on a model."""
    print(f"\n   🔬 Computing SHAP values for {model_name} ({ticker})...")
    
    # Sample data if too large
    if len(X_sample) > max_samples:
        idx = np.random.choice(len(X_sample), max_samples, replace=False)
        X_sample = X_sample[idx]
    
    # Create explainer
    if 'XGB' in model_name.upper():
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.TreeExplainer(model)
    
    shap_values = explainer.shap_values(X_sample)
    
    # Handle binary classification (get positive class SHAP values)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Positive class
    
    return explainer, shap_values, X_sample

# Run SHAP analysis
print("=" * 60)
print("🔬 SHAP ANALYSIS")
print("=" * 60)
print("\n⏳ This may take a minute...")

SHAP_RESULTS = {}

# Analyze Random Forest
if 'RandomForest' in CONFIG['models'] and RF_MODELS:
    for ticker, model in RF_MODELS.items():
        ds = DATASETS[ticker]
        explainer, shap_vals, X_sample = run_shap_analysis(
            model, ds['X_test'], engineer.feature_names, 'Random Forest', ticker
        )
        SHAP_RESULTS[f'RF_{ticker}'] = {
            'explainer': explainer,
            'shap_values': shap_vals,
            'X_sample': X_sample
        }
        
        # Summary plot
        print(f"\n   📊 SHAP Summary - Random Forest ({ticker}):")
        fig = plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_vals, X_sample, feature_names=engineer.feature_names, show=False)
        plt.tight_layout()
        plt.show()

# Analyze XGBoost
if 'XGBoost' in CONFIG['models'] and XGB_MODELS:
    for ticker, model in XGB_MODELS.items():
        ds = DATASETS[ticker]
        explainer, shap_vals, X_sample = run_shap_analysis(
            model, ds['X_test'], engineer.feature_names, 'XGBoost', ticker
        )
        SHAP_RESULTS[f'XGB_{ticker}'] = {
            'explainer': explainer,
            'shap_values': shap_vals,
            'X_sample': X_sample
        }
        
        # Summary plot
        print(f"\n   📊 SHAP Summary - XGBoost ({ticker}):")
        fig = plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_vals, X_sample, feature_names=engineer.feature_names, show=False)
        plt.tight_layout()
        plt.show()

print("\n✅ SHAP analysis complete!")
print("   💡 Tip: Red = high feature value, Blue = low feature value")
print("   💡 Position on X-axis shows impact on prediction (+ = Buy, - = Sell)")

---

## 📈 Section 7: Model Evaluation & Comparison

Now let's evaluate our models more thoroughly:
- Confusion matrix
- ROC curves
- Precision-Recall curves
- Simple backtesting simulation

> 💡 **For Beginners**: We compare models not just on accuracy, but on how well they'd work in real trading. A model that's accurate but loses money is useless!

In [ ]:
#@title 📊 Classification Metrics & Confusion Matrix { display-mode: "form" }
#@markdown Detailed classification performance analysis.

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score
import seaborn as sns
import numpy as np

def plot_confusion_matrix(y_true, y_pred, model_name, ticker):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Sell (0)', 'Buy (1)'],
                yticklabels=['Sell (0)', 'Buy (1)'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{model_name} Confusion Matrix ({ticker})')
    
    return fig

def plot_roc_curves(models_dict, predictions_dict, datasets, model_type):
    """Plot ROC curves for all tickers."""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(models_dict)))
    
    for i, (ticker, model) in enumerate(models_dict.items()):
        y_true = datasets[ticker]['y_test']
        y_prob = model.predict_proba(datasets[ticker]['X_test'])[:, 1]
        
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        
        ax.plot(fpr, tpr, color=colors[i], lw=2,
                label=f'{ticker} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_type} - ROC Curves')
    ax.legend(loc='lower right')
    
    return fig

# Generate evaluation plots
print("=" * 60)
print("📊 MODEL EVALUATION")
print("=" * 60)

# Confusion matrices
print("\n📋 Confusion Matrices:")

if 'RandomForest' in CONFIG['models'] and RF_PREDICTIONS:
    for ticker in RF_PREDICTIONS:
        y_true = DATASETS[ticker]['y_test']
        y_pred = RF_PREDICTIONS[ticker]
        fig = plot_confusion_matrix(y_true, y_pred, 'Random Forest', ticker)
        plt.show()
        
        print(f"\n   📝 Random Forest Classification Report ({ticker}):")
        print(classification_report(y_true, y_pred, target_names=['Sell', 'Buy']))

if 'XGBoost' in CONFIG['models'] and XGB_PREDICTIONS:
    for ticker in XGB_PREDICTIONS:
        y_true = DATASETS[ticker]['y_test']
        y_pred = XGB_PREDICTIONS[ticker]
        fig = plot_confusion_matrix(y_true, y_pred, 'XGBoost', ticker)
        plt.show()
        
        print(f"\n   📝 XGBoost Classification Report ({ticker}):")
        print(classification_report(y_true, y_pred, target_names=['Sell', 'Buy']))

# ROC Curves
print("\n📈 ROC Curves:")

if 'RandomForest' in CONFIG['models'] and RF_MODELS:
    fig = plot_roc_curves(RF_MODELS, RF_PREDICTIONS, DATASETS, 'Random Forest')
    plt.show()

if 'XGBoost' in CONFIG['models'] and XGB_MODELS:
    fig = plot_roc_curves(XGB_MODELS, XGB_PREDICTIONS, DATASETS, 'XGBoost')
    plt.show()

print("\n✅ Evaluation complete!")

In [ ]:
#@title 📈 Simple Backtesting Simulation { display-mode: "form" }
#@markdown Simulates trading performance using model predictions.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def simple_backtest(predictions, actual_returns, dates, model_name, ticker, initial_capital=10000):
    """
    Simple backtesting simulation.
    
    Strategy:
    - Buy (invest) when model predicts 1 (positive return)
    - Stay in cash when model predicts 0 (negative return)
    """
    n = len(predictions)
    
    # Portfolio value over time
    portfolio = np.zeros(n)
    portfolio[0] = initial_capital
    
    # Calculate returns
    for i in range(1, n):
        if predictions[i-1] == 1:  # Model said "buy"
            daily_return = actual_returns[i]
            portfolio[i] = portfolio[i-1] * (1 + daily_return)
        else:  # Model said "sell/hold"
            portfolio[i] = portfolio[i-1]  # Stay in cash
    
    # Buy and hold benchmark
    buy_hold = initial_capital * (1 + actual_returns).cumprod()
    buy_hold = np.insert(buy_hold[:-1], 0, initial_capital)
    
    # Calculate metrics
    total_return = (portfolio[-1] / portfolio[0]) - 1
    bh_return = (buy_hold[-1] / buy_hold[0]) - 1
    
    # Sharpe ratio (annualized)
    strategy_returns = np.diff(portfolio) / portfolio[:-1]
    sharpe = np.sqrt(252) * strategy_returns.mean() / (strategy_returns.std() + 1e-10)
    
    # Maximum drawdown
    peak = np.maximum.accumulate(portfolio)
    drawdown = (peak - portfolio) / peak
    max_drawdown = drawdown.max()
    
    # Win rate
    profitable_days = np.sum(strategy_returns > 0)
    total_trades = np.sum(predictions == 1)
    
    results = {
        'model_name': model_name,
        'ticker': ticker,
        'total_return': total_return,
        'buy_hold_return': bh_return,
        'sharpe_ratio': sharpe,
        'max_drawdown': max_drawdown,
        'total_trades': total_trades,
        'portfolio': portfolio,
        'buy_hold': buy_hold,
        'dates': dates
    }
    
    return results

def plot_backtest(results):
    """Plot backtest results."""
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Portfolio value
    ax1 = axes[0]
    ax1.plot(results['dates'], results['portfolio'], label='Strategy', linewidth=2)
    ax1.plot(results['dates'], results['buy_hold'], label='Buy & Hold', linewidth=2, alpha=0.7)
    ax1.set_title(f"{results['model_name']} Backtest ({results['ticker']})")
    ax1.set_ylabel('Portfolio Value ($)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Drawdown
    ax2 = axes[1]
    peak = np.maximum.accumulate(results['portfolio'])
    drawdown = (peak - results['portfolio']) / peak * 100
    ax2.fill_between(results['dates'], 0, -drawdown, alpha=0.5, color='red')
    ax2.set_ylabel('Drawdown (%)')
    ax2.set_xlabel('Date')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Run backtests
print("=" * 60)
print("📈 BACKTESTING SIMULATION")
print("=" * 60)

BACKTEST_RESULTS = {}

for ticker, ds in DATASETS.items():
    # Get actual returns for test period
    test_df = PROCESSED_DATA[ticker].iloc[-len(ds['y_test']):]
    actual_returns = test_df['returns'].values
    
    # Backtest Random Forest
    if 'RandomForest' in CONFIG['models'] and ticker in RF_PREDICTIONS:
        results = simple_backtest(
            RF_PREDICTIONS[ticker], actual_returns,
            ds['dates_test'], 'Random Forest', ticker
        )
        BACKTEST_RESULTS[f'RF_{ticker}'] = results
        
        print(f"\n🌲 Random Forest ({ticker}):")
        print(f"   • Strategy Return: {results['total_return']:.1%}")
        print(f"   • Buy & Hold Return: {results['buy_hold_return']:.1%}")
        print(f"   • Sharpe Ratio: {results['sharpe_ratio']:.2f}")
        print(f"   • Max Drawdown: {results['max_drawdown']:.1%}")
        
        fig = plot_backtest(results)
        plt.show()
    
    # Backtest XGBoost
    if 'XGBoost' in CONFIG['models'] and ticker in XGB_PREDICTIONS:
        results = simple_backtest(
            XGB_PREDICTIONS[ticker], actual_returns,
            ds['dates_test'], 'XGBoost', ticker
        )
        BACKTEST_RESULTS[f'XGB_{ticker}'] = results
        
        print(f"\n🚀 XGBoost ({ticker}):")
        print(f"   • Strategy Return: {results['total_return']:.1%}")
        print(f"   • Buy & Hold Return: {results['buy_hold_return']:.1%}")
        print(f"   • Sharpe Ratio: {results['sharpe_ratio']:.2f}")
        print(f"   • Max Drawdown: {results['max_drawdown']:.1%}")
        
        fig = plot_backtest(results)
        plt.show()

print("\n✅ Backtesting complete!")
print("   ⚠️ Note: This is a simplified simulation. Real trading involves")
print("      transaction costs, slippage, and other factors not modeled here.")

In [ ]:
#@title 🏆 Model Comparison Summary { display-mode: "form" }
#@markdown Compare all trained models side by side.

import pandas as pd
from IPython.display import display, HTML

def create_comparison_table():
    """Create a comparison table of all models."""
    rows = []
    
    # Random Forest results
    if 'RandomForest' in CONFIG['models'] and RF_METRICS:
        for ticker, metrics in RF_METRICS.items():
            backtest = BACKTEST_RESULTS.get(f'RF_{ticker}', {})
            rows.append({
                'Model': 'Random Forest',
                'Ticker': ticker,
                'Train Acc': f"{metrics['train_accuracy']:.3f}",
                'Test Acc': f"{metrics['test_accuracy']:.3f}",
                'CV Mean': f"{metrics['cv_mean']:.3f}" if metrics['cv_mean'] else 'N/A',
                'F1 Score': f"{metrics['test_f1']:.3f}",
                'Strategy Return': f"{backtest.get('total_return', 0):.1%}",
                'Sharpe': f"{backtest.get('sharpe_ratio', 0):.2f}",
                'Max DD': f"{backtest.get('max_drawdown', 0):.1%}",
                'Train Time': f"{metrics['train_time']:.1f}s"
            })
    
    # XGBoost results
    if 'XGBoost' in CONFIG['models'] and XGB_METRICS:
        for ticker, metrics in XGB_METRICS.items():
            backtest = BACKTEST_RESULTS.get(f'XGB_{ticker}', {})
            rows.append({
                'Model': 'XGBoost',
                'Ticker': ticker,
                'Train Acc': f"{metrics['train_accuracy']:.3f}",
                'Test Acc': f"{metrics['test_accuracy']:.3f}",
                'CV Mean': f"{metrics['cv_mean']:.3f}" if metrics['cv_mean'] else 'N/A',
                'F1 Score': f"{metrics['test_f1']:.3f}",
                'Strategy Return': f"{backtest.get('total_return', 0):.1%}",
                'Sharpe': f"{backtest.get('sharpe_ratio', 0):.2f}",
                'Max DD': f"{backtest.get('max_drawdown', 0):.1%}",
                'Train Time': f"{metrics['train_time']:.1f}s"
            })
    
    return pd.DataFrame(rows)

# Display comparison
print("=" * 60)
print("🏆 MODEL COMPARISON SUMMARY")
print("=" * 60)

comparison_df = create_comparison_table()

if len(comparison_df) > 0:
    # Style the dataframe
    display(comparison_df.style.set_properties(**{
        'text-align': 'center',
        'font-size': '12px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]}
    ]))
    
    # Find best model
    print("\n📊 Analysis:")
    
    # Best by test accuracy
    if len(comparison_df) > 1:
        best_acc_idx = comparison_df['Test Acc'].apply(lambda x: float(x)).idxmax()
        best_acc = comparison_df.iloc[best_acc_idx]
        print(f"   • Best Test Accuracy: {best_acc['Model']} ({best_acc['Ticker']}) - {best_acc['Test Acc']}")
        
        # Note about overfitting
        for idx, row in comparison_df.iterrows():
            train = float(row['Train Acc'])
            test = float(row['Test Acc'])
            if train - test > 0.1:
                print(f"   ⚠️ Possible overfitting: {row['Model']} ({row['Ticker']}) - Train: {row['Train Acc']}, Test: {row['Test Acc']}")
else:
    print("❌ No models trained yet.")

print("\n✅ Comparison complete!")

---

## 📦 Section 8: Export & Experiment Tracking

Final steps:
1. **Export to ONNX** - Portable model format for dashboard deployment
2. **Log to Weights & Biases** - Track experiments over time
3. **Log to Dashboard API** - Register model for paper trading

### Why ONNX?

| Format | Pros | Cons |
|--------|------|------|
| **Pickle** | Python native, simple | Python-only, security risks |
| **Joblib** | Better for large arrays | Python-only |
| **ONNX** | Cross-platform, production-ready | Conversion complexity |

> 💡 **For Beginners**: ONNX (Open Neural Network Exchange) lets your model run anywhere - Python, JavaScript, mobile apps, or web servers!

In [ ]:
#@title 📦 Export to ONNX { display-mode: "form" }
#@markdown Converts trained models to ONNX format for deployment.

from skl2onnx import convert_sklearn, to_onnx
from skl2onnx.common.data_types import FloatTensorType
import onnx
import os
import numpy as np

def export_to_onnx(model, n_features, model_name, ticker, output_dir):
    """Export a sklearn/xgboost model to ONNX format."""
    
    # Define input type
    initial_type = [('float_input', FloatTensorType([None, n_features]))]
    
    try:
        # Convert to ONNX
        if 'XGB' in model_name.upper():
            # XGBoost needs special handling
            onnx_model = convert_sklearn(
                model, initial_types=initial_type,
                target_opset=12,
                options={type(model): {'zipmap': False}}
            )
        else:
            # Random Forest
            onnx_model = convert_sklearn(
                model, initial_types=initial_type,
                target_opset=12
            )
        
        # Save to file
        filename = f"{model_name.lower().replace(' ', '_')}_{ticker}.onnx"
        filepath = os.path.join(output_dir, filename)
        onnx.save_model(onnx_model, filepath)
        
        return filepath, onnx_model
    
    except Exception as e:
        print(f"      ❌ Error exporting {model_name} ({ticker}): {e}")
        return None, None

# Export all models
print("=" * 60)
print("📦 EXPORTING TO ONNX")
print("=" * 60)

ONNX_DIR = os.path.join(CHECKPOINT_DIR, 'onnx_models')
os.makedirs(ONNX_DIR, exist_ok=True)

ONNX_EXPORTS = {}
n_features = len(engineer.feature_names)

# Export Random Forest models
if 'RandomForest' in CONFIG['models'] and RF_MODELS:
    print("\n🌲 Exporting Random Forest models:")
    for ticker, model in RF_MODELS.items():
        print(f"   • {ticker}...", end=" ")
        filepath, onnx_model = export_to_onnx(
            model, n_features, 'random_forest', ticker, ONNX_DIR
        )
        if filepath:
            ONNX_EXPORTS[f'RF_{ticker}'] = filepath
            print(f"✅ Saved to {os.path.basename(filepath)}")

# Export XGBoost models
if 'XGBoost' in CONFIG['models'] and XGB_MODELS:
    print("\n🚀 Exporting XGBoost models:")
    for ticker, model in XGB_MODELS.items():
        print(f"   • {ticker}...", end=" ")
        filepath, onnx_model = export_to_onnx(
            model, n_features, 'xgboost', ticker, ONNX_DIR
        )
        if filepath:
            ONNX_EXPORTS[f'XGB_{ticker}'] = filepath
            print(f"✅ Saved to {os.path.basename(filepath)}")

print(f"\n✅ ONNX export complete!")
print(f"   📁 Models saved to: {ONNX_DIR}")
print(f"   📊 Total models exported: {len(ONNX_EXPORTS)}")

In [ ]:
#@title 📊 Log to Weights & Biases { display-mode: "form" }
#@markdown Track experiments in W&B for long-term analysis.

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("⚠️ wandb not installed. Run: pip install wandb")

def log_to_wandb(experiment_name, config, metrics_dict, backtest_results, model_type):
    """Log experiment to Weights & Biases."""
    
    if not WANDB_AVAILABLE:
        return None
    
    # Initialize W&B run
    run = wandb.init(
        project="trading-ml-tree-models",
        name=f"{experiment_name}_{model_type}",
        config={
            **config,
            'model_type': model_type,
            'notebook': 'tree_based'
        },
        reinit=True
    )
    
    # Log metrics for each ticker
    for ticker, metrics in metrics_dict.items():
        wandb.log({
            f"{ticker}/train_accuracy": metrics['train_accuracy'],
            f"{ticker}/test_accuracy": metrics['test_accuracy'],
            f"{ticker}/test_precision": metrics['test_precision'],
            f"{ticker}/test_recall": metrics['test_recall'],
            f"{ticker}/test_f1": metrics['test_f1'],
            f"{ticker}/cv_mean": metrics['cv_mean'] or 0,
            f"{ticker}/train_time": metrics['train_time']
        })
        
        # Log backtest results
        bt_key = f"{model_type[:2].upper()}_{ticker}"
        if bt_key in backtest_results:
            bt = backtest_results[bt_key]
            wandb.log({
                f"{ticker}/strategy_return": bt['total_return'],
                f"{ticker}/sharpe_ratio": bt['sharpe_ratio'],
                f"{ticker}/max_drawdown": bt['max_drawdown']
            })
    
    wandb.finish()
    return run

# Log to W&B
print("=" * 60)
print("📊 LOGGING TO WEIGHTS & BIASES")
print("=" * 60)

if WANDB_AVAILABLE:
    try:
        # Log Random Forest
        if 'RandomForest' in CONFIG['models'] and RF_METRICS:
            print("\n🌲 Logging Random Forest experiment...")
            log_to_wandb(
                CONFIG['experiment_name'],
                CONFIG,
                RF_METRICS,
                BACKTEST_RESULTS,
                'RandomForest'
            )
            print("   ✅ Random Forest logged to W&B")
        
        # Log XGBoost
        if 'XGBoost' in CONFIG['models'] and XGB_METRICS:
            print("\n🚀 Logging XGBoost experiment...")
            log_to_wandb(
                CONFIG['experiment_name'],
                CONFIG,
                XGB_METRICS,
                BACKTEST_RESULTS,
                'XGBoost'
            )
            print("   ✅ XGBoost logged to W&B")
        
        print("\n✅ W&B logging complete!")
        print("   📊 View at: https://wandb.ai")
        
    except Exception as e:
        print(f"\n⚠️ W&B logging failed: {e}")
        print("   💡 Tip: Run `wandb login` to authenticate")
else:
    print("\n⚠️ Weights & Biases not available.")
    print("   Install with: pip install wandb")

In [ ]:
#@title 📡 Log to Dashboard API { display-mode: "form" }
#@markdown Register model with the trading dashboard for paper trading.

import requests
import json
from datetime import datetime

DASHBOARD_API_URL = os.getenv('DASHBOARD_API_URL', 'http://localhost:3000/api')

def log_to_dashboard(experiment_name, model_type, ticker, metrics, onnx_path, config):
    """Register model with the dashboard API."""
    
    payload = {
        'name': f"{experiment_name}_{model_type}_{ticker}",
        'type': model_type.lower(),
        'ticker': ticker,
        'metrics': {
            'accuracy': metrics['test_accuracy'],
            'precision': metrics['test_precision'],
            'recall': metrics['test_recall'],
            'f1_score': metrics['test_f1'],
            'cv_score': metrics['cv_mean'] or 0
        },
        'config': {
            'n_estimators': config.get(f'{model_type.lower()[:2]}_n_estimators', 
                                      config.get('rf_n_estimators', 100)),
            'max_depth': config.get(f'{model_type.lower()[:2]}_max_depth',
                                   config.get('rf_max_depth', 10)),
            'features': len(config.get('tickers', [])),
            'train_split': config.get('train_split', 0.8)
        },
        'onnx_path': onnx_path,
        'notebook_type': 'tree_based',
        'created_at': datetime.now().isoformat()
    }
    
    try:
        response = requests.post(
            f"{DASHBOARD_API_URL}/models",
            json=payload,
            timeout=10
        )
        
        if response.status_code in [200, 201]:
            return response.json()
        else:
            print(f"      ⚠️ API returned status {response.status_code}")
            return None
            
    except requests.exceptions.ConnectionError:
        print(f"      ⚠️ Could not connect to dashboard API")
        return None
    except Exception as e:
        print(f"      ⚠️ Error: {e}")
        return None

# Log to dashboard
print("=" * 60)
print("📡 LOGGING TO DASHBOARD API")
print("=" * 60)

API_RESULTS = {}

# Log Random Forest models
if 'RandomForest' in CONFIG['models'] and RF_METRICS:
    print("\n🌲 Registering Random Forest models:")
    for ticker, metrics in RF_METRICS.items():
        print(f"   • {ticker}...", end=" ")
        onnx_path = ONNX_EXPORTS.get(f'RF_{ticker}', '')
        result = log_to_dashboard(
            CONFIG['experiment_name'], 'RandomForest', ticker,
            metrics, onnx_path, CONFIG
        )
        if result:
            API_RESULTS[f'RF_{ticker}'] = result
            print("✅")
        else:
            print("⚠️ (will retry later)")

# Log XGBoost models
if 'XGBoost' in CONFIG['models'] and XGB_METRICS:
    print("\n🚀 Registering XGBoost models:")
    for ticker, metrics in XGB_METRICS.items():
        print(f"   • {ticker}...", end=" ")
        onnx_path = ONNX_EXPORTS.get(f'XGB_{ticker}', '')
        result = log_to_dashboard(
            CONFIG['experiment_name'], 'XGBoost', ticker,
            metrics, onnx_path, CONFIG
        )
        if result:
            API_RESULTS[f'XGB_{ticker}'] = result
            print("✅")
        else:
            print("⚠️ (will retry later)")

print(f"\n✅ Dashboard logging complete!")
print(f"   📊 Models registered: {len(API_RESULTS)}")
if not API_RESULTS:
    print("   💡 Tip: Make sure the dashboard server is running")

---

## 🎉 Section 9: Summary & Next Steps

Congratulations! You've completed the tree-based models training pipeline.

### What You've Accomplished:

✅ **Data Pipeline**
- Fetched historical market data
- Created 30+ technical indicators
- Time-series aware train/test split

✅ **Model Training**
- Random Forest with cross-validation
- XGBoost with early stopping
- Hyperparameter configuration via widgets

✅ **Feature Analysis**
- Built-in feature importance
- SHAP value analysis
- Identified key predictive features

✅ **Evaluation**
- Classification metrics (accuracy, precision, recall, F1)
- ROC curves and confusion matrices
- Backtesting simulation

✅ **Deployment Ready**
- ONNX export for production
- W&B experiment tracking
- Dashboard API integration

### Next Steps:

1. **🔬 Hyperparameter Tuning**: Use Optuna for automated optimization
2. **📊 Compare with Neural Networks**: Open `neural_networks_training.ipynb`
3. **💰 Paper Trading**: Deploy to dashboard for simulated trading
4. **🔄 Iterate**: Try different features, tickers, or time periods

### Cross-Notebook Tips:

- Your checkpoints are saved in the shared folder
- Neural network notebook can load your preprocessed data
- Compare tree-based vs. neural network performance on the same data

---

**Need Help?** Check the [README](./README.md) or reach out to the team!

*Happy Trading! 🚀📈*

In [ ]:
#@title 📋 Final Experiment Summary { display-mode: "form" }
#@markdown Generates a complete summary of your experiment session.

from datetime import datetime
from IPython.display import display, Markdown

def generate_summary():
    """Generate a comprehensive experiment summary."""
    
    summary = f"""
# 📋 Experiment Summary

**Experiment Name:** `{CONFIG['experiment_name']}`  
**Timestamp:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Notebook:** Tree-Based Models Training

---

## 📊 Data Configuration

| Setting | Value |
|---------|-------|
| Data Source | {CONFIG['data_source']} |
| Tickers | {', '.join(CONFIG['tickers'])} |
| Date Range | {CONFIG['start_date']} to {CONFIG['end_date']} |
| Train/Test Split | {CONFIG['train_split']:.0%}/{1-CONFIG['train_split']:.0%} |
| CV Folds | {CONFIG['cv_folds']} |
| Features | {len(engineer.feature_names) if 'engineer' in dir() else 'N/A'} |

---

## 🏆 Model Results

"""
    
    # Random Forest results
    if 'RandomForest' in CONFIG['models'] and RF_METRICS:
        summary += "### 🌲 Random Forest\n\n"
        summary += "| Ticker | Test Accuracy | F1 Score | Strategy Return | Sharpe |\n"
        summary += "|--------|--------------|----------|-----------------|--------|\n"
        
        for ticker, metrics in RF_METRICS.items():
            bt = BACKTEST_RESULTS.get(f'RF_{ticker}', {})
            summary += f"| {ticker} | {metrics['test_accuracy']:.3f} | {metrics['test_f1']:.3f} | "
            summary += f"{bt.get('total_return', 0):.1%} | {bt.get('sharpe_ratio', 0):.2f} |\n"
        
        summary += f"\n**Config:** n_estimators={CONFIG['rf_n_estimators']}, "
        summary += f"max_depth={CONFIG['rf_max_depth']}, min_samples_split={CONFIG['rf_min_samples_split']}\n\n"
    
    # XGBoost results
    if 'XGBoost' in CONFIG['models'] and XGB_METRICS:
        summary += "### 🚀 XGBoost\n\n"
        summary += "| Ticker | Test Accuracy | F1 Score | Strategy Return | Sharpe |\n"
        summary += "|--------|--------------|----------|-----------------|--------|\n"
        
        for ticker, metrics in XGB_METRICS.items():
            bt = BACKTEST_RESULTS.get(f'XGB_{ticker}', {})
            summary += f"| {ticker} | {metrics['test_accuracy']:.3f} | {metrics['test_f1']:.3f} | "
            summary += f"{bt.get('total_return', 0):.1%} | {bt.get('sharpe_ratio', 0):.2f} |\n"
        
        summary += f"\n**Config:** n_estimators={CONFIG['xgb_n_estimators']}, "
        summary += f"max_depth={CONFIG['xgb_max_depth']}, learning_rate={CONFIG['xgb_learning_rate']}, "
        summary += f"subsample={CONFIG['xgb_subsample']}\n\n"
    
    summary += f"""
---

## 📁 Saved Artifacts

| Artifact | Location |
|----------|----------|
| Checkpoint | `{CHECKPOINT_DIR}` |
| ONNX Models | `{ONNX_DIR if 'ONNX_DIR' in dir() else 'N/A'}` |
| Models Exported | {len(ONNX_EXPORTS) if 'ONNX_EXPORTS' in dir() else 0} |

---

## 🔄 Reproducibility

To reproduce this experiment:

```python
CONFIG = {json.dumps(CONFIG, indent=2, default=str)}
```

---

*Generated by Tree-Based Models Training Notebook*
"""
    
    return summary

# Display summary
print("=" * 60)
print("📋 GENERATING FINAL SUMMARY")
print("=" * 60)

summary_md = generate_summary()
display(Markdown(summary_md))

print("\n✅ Experiment complete! 🎉")